In [3]:
import os
import numpy as np
import faiss
import torch
import re
from sentence_transformers import SentenceTransformer
from collections import Counter
from tqdm import tqdm

# 1 Semantic Processor (The "Signal" Extractor)

def semantic_puml_processor(text):
    """
    Strips noise and highlights DP roles and Relationships.
    Transforms raw PUML into a format the BGE model can actually understand.
    """
    processed_tokens = []
    
    # Extract DP Roles from comments
    roles = re.findall(r"'\s*DP role:\s*(\w+)", text, re.IGNORECASE)
    processed_tokens.extend([f"role_{r.lower()}" for r in roles])
    
     # 2. Extract Class/Interface names and simplify namespaces
    classes = re.findall(r"(?:class|interface)\s+([\w\.]+)", text)
    for c in classes:
        short_name = c.split('.')[-1].lower()
        processed_tokens.append(short_name)
 
    # 3. Convert symbolic relationships to semantic words
    rel_map = {
        r"<\|--": "inheritance",
        r"<\|..": "implementation",
        r"-->": "association",
        r"\*--": "composition"
    }
    for symbol, word in rel_map.items():
        if re.search(symbol, text):
            processed_tokens.append(word)

    # 4. Filter out Java primitives and generic boilerplate
    noise = {}
    final_tokens = [t for t in processed_tokens if t not in noise]

    return " ".join(final_tokens)

##############################################
# 2. Updated Loading & Indexing
##############################################

def load_and_process_docs(folder_path):
    documents = []
    for file in os.listdir(folder_path):
        if file.endswith((".puml", ".plantuml")):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                content = f.read()
                # Apply semantic processing during load
                semantic_content = semantic_puml_processor(content)
                
                if len(semantic_content.strip()) > 0:
                    documents.append({
                        "id": file,
                        "content": semantic_content,
                        "raw": content
                    })
    return documents

class EmbeddingModel:
    def __init__(self, model_name="BAAI/bge-base-en"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        # trust_remote_code=True is required for some BGE versions
        self.model = SentenceTransformer(model_name)
        self.model = self.model.to(self.device)

    def encode(self, texts, batch_size=16):
        return self.model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True, # Critical for Cosine Similarity in FAISS
            show_progress_bar=True
        )

class VectorStore:
    def __init__(self, embedding_dim):
        # IndexFlatIP calculates Inner Product, which is Cosine Similarity for normalized vectors
        self.index = faiss.IndexFlatIP(embedding_dim)
        self.metadata = []
    
    def add_embeddings(self, embeddings, metadatas):
        self.index.add(embeddings)
        self.metadata.extend(metadatas)
    
    def search(self, query_embedding, top_k=5):
        distances, indices = self.index.search(query_embedding, top_k)
        results = []
        for i, idx in enumerate(indices[0]):
            if idx != -1:
                res = self.metadata[idx].copy()
                res['score'] = distances[0][i] # Keep track of similarity
                results.append(res)
        return results

##############################################
# 3. Main Pipeline Functions
##############################################

def build_rag_index(folder_path):
    print("Loading and processing PlantUML files...")
    docs = load_and_process_docs(folder_path)
    
    texts = [doc["content"] for doc in docs]
    
    embedder = EmbeddingModel()
    embeddings = embedder.encode(texts)

    vector_store = VectorStore(embedding_dim=embeddings.shape[1])
    vector_store.add_embeddings(np.array(embeddings), docs)

    return vector_store, embedder

def detect_pattern(test_doc, vector_store, embedder, top_k=7):
    # Ensure the test document is also semantically processed
    query_text = semantic_puml_processor(test_doc["content"])
    query_embedding = embedder.encode([query_text])
    
    results = vector_store.search(query_embedding, top_k=top_k)

    # Weighted Voting: High similarity matches have more influence
    votes = {}
    for r in results:
        label = r["id"].split('-')[0].split('_')[0] # Robust label extraction
        votes[label] = votes.get(label, 0) + r['score']

    predicted_label = max(votes, key=votes.get) if votes else "Unknown"
    retrieved_neighbors = [r["id"] for r in results]
    
    return predicted_label, retrieved_neighbors

def evaluate_detection(test_folder, vector_store, embedder, top_k=7):
    test_docs = load_and_process_docs(test_folder)
    
    # Counters for per-pattern accuracy
    correct_counts = Counter()
    total_counts = Counter()
    
    for doc in test_docs:
        # Ground truth from filename
        true_label = doc["id"].split('-')[0].split('_')[0]
        total_counts[true_label] += 1
        
        predicted_label, neighbors = detect_pattern(
            {"content": doc["raw"]},  # Pass raw content
            vector_store,
            embedder,
            top_k=top_k
        )

        if predicted_label == true_label:
            correct_counts[true_label] += 1

        print(f"File: {doc['id']} | True: {true_label} | Pred: {predicted_label}")

    # Compute per-pattern accuracy
    per_pattern_accuracy = {}
    for label in total_counts:
        per_pattern_accuracy[label] = correct_counts[label] / total_counts[label]

    # Overall accuracy
    overall_accuracy = sum(correct_counts.values()) / sum(total_counts.values()) if total_counts else 0

    print("\nPer-Pattern Accuracy:")
    for label, acc in per_pattern_accuracy.items():
        print(f"  {label}: {acc:.4f}")

    print(f"\nOverall Accuracy: {overall_accuracy:.4f}")
    return overall_accuracy, per_pattern_accuracy

In [4]:
# 1. Build the vector store and embedding model from your knowledge base
vector_store, embedder = build_rag_index("PMART_PUML/knowledge_base")

# 2. Run evaluation on test folder directly
accuracy, per_pattern_accuracy = evaluate_detection(
    test_folder="PMART_PUML/test",  # Folder path is passed directly
    vector_store=vector_store,
    embedder=embedder,
    top_k=2  # Number of neighbors to retrieve for LLM context
)

print(f"\nFinal Overall Accuracy: {accuracy:.4f}")
print("Per-Pattern Accuracy Details:")
for pattern, acc in per_pattern_accuracy.items():
    print(f"  {pattern}: {acc:.4f}")

Loading and processing PlantUML files...


Batches: 100%|██████████| 1/1 [00:00<00:00, 220.60it/s]


File: Adapter-430.puml | True: Adapter | Pred: Adapter


Batches: 100%|██████████| 1/1 [00:00<00:00, 253.74it/s]


File: Template Method-2222.puml | True: Template Method | Pred: Proxy


...

File: Proxy-2040.puml | True: Proxy | Pred: Proxy


Batches: 100%|██████████| 1/1 [00:00<00:00, 217.19it/s]

File: Abstract Factory-42.puml | True: Abstract Factory | Pred: Abstract Factory



Batches: 100%|██████████| 1/1 [00:00<00:00, 252.24it/s]

File: Adapter-52.puml | True: Adapter | Pred: Adapter

Per-Pattern Accuracy:
  Adapter: 0.8571
  Template Method: 0.0000
  Abstract Factory: 1.0000
  Singleton: 0.4286
  Strategy: 0.5000
  State: 0.3333
  Iterator: 1.0000
  Command: 0.0000
  Visitor: 0.5000
  Proxy: 1.0000
  Factory Method: 0.0000

Overall Accuracy: 0.5811

Final Overall Accuracy: 0.5811
Per-Pattern Accuracy Details:
  Adapter: 0.8571
  Template Method: 0.0000
  Abstract Factory: 1.0000
  Singleton: 0.4286
  Strategy: 0.5000
  State: 0.3333
  Iterator: 1.0000
  Command: 0.0000
  Visitor: 0.5000
  Proxy: 1.0000
  Factory Method: 0.0000
